# core

> Gateway naming, startup delivery, and MCP sessions on rustygate gateways

In [ ]:
#| default_exp core

clikernel is the LLM side of a two-process design: a gateway ([rustygate](https://github.com/AnswerDotAI/rustygate)) hosts the kernels and serves MCP itself at `POST /mcp`; clikernel starts and stops with each conversation and routes the harness's stdio MCP to gateways. This module is the client layer: gateway naming from `gateways.toml`, the per-session kernel-creation defaults (the conversation's cwd and environment, with `startup.py` and `inspectors.py` composed into one startup source), `Gateway` — one MCP session on one gateway — and `default_gateway`, which finds the local gateway or starts an owned child that lives exactly as long as the conversation. Kernel lifecycle policy lives gateway-side: ending a session stops the kernels it created with autoclose and nothing else.


In [ ]:
#| export
import os, tomllib, httpx
from fastcore.utils import *
from fastcore.xdg import xdg_config_home
from mcpmini.core import HTTPTransport, jreq
from rustygate.tools import start_gateway
from clikernel import __version__

In [ ]:
from fastcore.test import *
import asyncio, tempfile
import re


## Configuration

In [ ]:
#| export
DEFAULT_URL = 'http://127.0.0.1:8787'

def cfg_dir():
    "The clikernel config directory"
    return xdg_config_home()/'clikernel'

def gateways(cfgdir=None):
    "Named gateways from `gateways.toml`: `{name: {url, token | token_env, verify}}`"
    p = (Path(cfgdir) if cfgdir else cfg_dir())/'gateways.toml'
    return tomllib.loads(p.read_text()).get('gateways', {}) if p.exists() else {}

def resolve(host='', cfgdir=None):
    "`(url, token, verify)` for `host`: empty = the default local gateway, a URL = itself, else a `gateways.toml` name"
    if not host: return os.environ.get('CLIKERNEL_HOST', DEFAULT_URL), os.environ.get('CLIKERNEL_TOKEN'), True
    if '://' in host: return host, os.environ.get('CLIKERNEL_TOKEN'), True
    cfgdir = Path(cfgdir) if cfgdir else cfg_dir()
    g = gateways(cfgdir).get(host)
    if g is None: raise ValueError(f"unknown gateway {host!r}: not a URL, and not in {cfgdir/'gateways.toml'}")
    return g['url'], g.get('token') or os.environ.get(g.get('token_env','')) or None, g.get('verify', True)


`$XDG_CONFIG_HOME/clikernel/` holds three files, all optional: `startup.py` (run in every Python kernel a clikernel session creates), `inspectors.py` (installed in the same breath), and `gateways.toml`, which names remote gateways so that tokens never travel as tool arguments (tool args persist in transcripts). A `host` is resolved in one of three ways: empty means the default local gateway (`$CLIKERNEL_HOST` or `http://127.0.0.1:8787`), a URL is used as given, and anything else is looked up as a `gateways.toml` name:

    [gateways.solveit]
    url = "https://solveit.example.com/gate"
    token_env = "SOLVEIT_TOKEN"
    verify = false   # accept a self-signed certificate (e.g. rustygate --tls)


In [ ]:
cfgd = Path(tempfile.mkdtemp())
(cfgd/'gateways.toml').write_text('[gateways.solveit]\nurl = "https://s.example.com/gate"\ntoken = "T"\nverify = false\n')
test_eq(resolve(), (DEFAULT_URL, os.environ.get('CLIKERNEL_TOKEN'), True))
test_eq(resolve('http://h:1/p'), ('http://h:1/p', os.environ.get('CLIKERNEL_TOKEN'), True))
test_eq(resolve('solveit', cfgd), ('https://s.example.com/gate', 'T', False))
test_fail(lambda: resolve('nope', cfgd), contains=str(cfgd/'gateways.toml'))


## Startup and inspectors


Both config files travel *as source*, composed into one startup program that the gateway runs in each Python kernel this session creates, before any user code — the kernel may be on another machine, so no file path can be assumed there, and the local config stays authoritative either way. `startup.py` runs first, wrapped so `__file__` is bound to its local path during the run and gone afterwards (matching v1, which used `%run -i`); its output comes back in the reply that announces the kernel. `inspectors.py` installs second, with v1's contract intact: the file may define `inspect` and/or a list `inspectors`; each is called once per cell before it runs — 1-arg inspectors get the cell's AST, 2-arg ones get `(tree, src)` with the raw source for lexical checks. An inspector may return a note (printed before the cell's output), raise `RuleBlock` (provided in the file's namespace; the cell does not run), or return None. Any other exception is an inspector bug: noted, and the cell runs — fail-open, because a crashed inspector must never masquerade as a policy block. A file that fails to *load* raises inside the startup program, so the gateway stops the fresh kernel and fails the creating call: refusing to start beats running uninspected.

The gateway never runs this Python startup or inspector program in Luau. The same cwd/environment defaults still apply; language selection belongs to the gateway's `py`/`lua` tools and `create(language=...)` parameter, not to the router.


In [ ]:
#| export
def _startup_src(src, path):
    "The startup file's source wrapped so `__file__` is bound to its path during the run, and absent after"
    return f'''__file__ = {str(path)!r}
try: exec(compile({src!r}, __file__, 'exec'))
finally: del __file__'''

In [ ]:
#| export
_INSP_RUNNER = r'''
import inspect as _clik_inspect
import sys as _clik_sys
from IPython.core.error import InputRejected
class RuleBlock(InputRejected):
    "Raise from an inspector to deliberately block a cell; any other inspector exception is a bug, and fails open"

class _ClikInspect:
    "Calls each inspector once per cell: 1-arg get the AST, 2-arg also the raw source"
    def __init__(self, fs): self.fs = fs
    def visit(self, tree):
        fr, n = _clik_sys._getframe(), 0
        while fr:
            n += fr.f_code.co_name == 'run_cell_async'
            fr = fr.f_back
        if n > 1: return tree  # nested run_cell: cell replayed by a tool (%nbrun etc.), not typed
        for f in self.fs:
            try:
                note = f(tree, _clik_src) if len(_clik_inspect.signature(f).parameters) > 1 else f(tree)
                if note: print(note, end='')
            except InputRejected: raise
            except Exception as e: print(f'inspector error (cell runs anyway): {e!r}')
        return tree

def _clik_stash(info):
    global _clik_src
    _clik_src = info.raw_cell

def _clik_install(src):
    ns = dict(RuleBlock=RuleBlock)
    exec(compile(src, 'inspectors.py', 'exec'), ns)
    fs = list(ns.get('inspectors') or [])
    if callable(ns.get('inspect')): fs.append(ns['inspect'])
    if fs:
        ip = get_ipython()
        ip.events.register('pre_run_cell', _clik_stash)
        ip.ast_transformers.append(_ClikInspect(fs))
_clik_src = ''
'''

def _inspector_setup(src):
    "Kernel-side source installing the inspectors defined in `src`; a load failure raises, failing the create call"
    return _INSP_RUNNER + f'\n_clik_install({src!r})'

`startup_src` composes the two files into the one program a gateway runs, and `session_defaults` packages it with the conversation's context:

In [ ]:
#| export
def startup_src(cfgdir=None):
    "The composed startup source for one kernel: `startup.py` wrapped, then the inspector installer"
    d = Path(cfgdir) if cfgdir else cfg_dir()
    parts = []
    if (p := d/'startup.py').exists(): parts.append(_startup_src(p.read_text(), p))
    if (p := d/'inspectors.py').exists(): parts.append(_inspector_setup(p.read_text()))
    return '\n'.join(parts)

def session_defaults(cfgdir=None, quiet=False, local=True):
    "The `rustygate` initialize extension: startup source and quiet, plus cwd and env for a local gateway"
    d = dict(startup=startup_src(cfgdir), quiet=quiet)
    if local:
        d['cwd'] = os.getcwd()
        d['env'] = dict(os.environ, CLIKERNEL_QUIET='1') if quiet else dict(os.environ)
    return d

`session_defaults` is everything a gateway needs to make kernels for this conversation, in the shape rustygate's initialize accepts. The conversation's cwd and environment only make sense on the same machine, so a named host gets neither — local paths and env vars mean nothing there. `CLIKERNEL_QUIET` rides in the env so kernel-side tooling can hush narration of its own (llmdojo's rule warnings do).

In [ ]:
cfgd = Path(tempfile.mkdtemp())
(cfgd/'startup.py').write_text('import sys\nbase = 42\nprint("ready, file", __file__.rsplit("/",1)[-1])')
insp_src = '''
import ast
def note_sleep(tree, src):
    if 'time.sleep' in src: return 'note: sleeping\\n'
def block_ctypes(tree):
    if any(isinstance(n, ast.Import) and any(a.name=='ctypes' for a in n.names) for n in ast.walk(tree)):
        raise RuleBlock('no ctypes in kernels')
def buggy(tree, src):
    if 'trigger_bug' in src: raise TypeError('oops')
inspectors = [note_sleep, block_ctypes, buggy]
'''
(cfgd/'inspectors.py').write_text(insp_src)
d = session_defaults(cfgd)
assert 'ready, file' in d['startup'] and '_clik_install' in d['startup']
test_eq(session_defaults(cfgd, local=False).keys(), {'startup', 'quiet'})
sorted(d)

['cwd', 'env', 'quiet', 'startup']

## The gateway session

`Gateway` is one MCP session on one rustygate: an `initialize` carrying the session defaults, tool calls, and a DELETE at close. The gateway holds every pointer that matters — the session's current kernel, which kernels autoclose — so this class is a wire surface, not a state holder. A token travels as a standard bearer token, and `verify=False` accepts a self-signed certificate. `text` is the common read: a tool reply's text blocks joined, raising on `isError` so lessons and callers read straight prose.

In [ ]:
#| export
class Gateway:
    "One MCP session on one rustygate: initialize with session defaults, call tools, DELETE at close"
    def __init__(self,
        url,          # The gateway base URL, e.g. 'http://127.0.0.1:8787'
        token=None,   # Gateway auth token, sent as a bearer token
        verify=True,  # Verify TLS certificates?
    ):
        client = httpx.AsyncClient(verify=verify, timeout=httpx.Timeout(None, connect=10))
        self.url,self._id = url,0
        self.tr = HTTPTransport(f"{url.rstrip('/')}/mcp", token=token, http_client=client)

    async def rpc(self, method, **params):
        "One JSON-RPC request, returning its result and raising on a protocol-level error"
        self._id += 1
        r = await self.tr.send(jreq(method, self._id, **params))
        if 'error' in r: raise RuntimeError(f"{r['error']['code']}: {r['error']['message']}")
        return r['result']

    async def initialize(self, defaults=None):
        "Open the MCP session, sending `defaults` as the `rustygate` extension; returns self"
        await self.tr.start()
        self.info = await self.rpc('initialize', protocolVersion='2025-11-25', capabilities={},
            clientInfo=dict(name='clikernel', version=__version__), rustygate=defaults or {})
        await self.tr.send(jreq('notifications/initialized'))
        return self

    async def tools(self): return (await self.rpc('tools/list'))['tools']
    async def call(self, name, **args): return await self.rpc('tools/call', name=name, arguments=args)

    async def text(self, name, **args):
        "A tool call's text blocks joined; raises on `isError`"
        r = await self.call(name, **args)
        t = ''.join(c.get('text','') for c in r['content'] if c['type'] == 'text')
        if r.get('isError'): raise RuntimeError(t)
        return t

    async def aclose(self):
        "End the MCP session — the gateway stops the kernels this session created with autoclose — and drop the connection"
        await self.tr.delete()
        await self.tr.aclose()

All of it live, against a rustygate on its own port with the config dir from above. The first `py` finds no current kernel, so the gateway creates one, runs the composed startup in it, and the reply opens with the kernel id line and the startup banner — `__file__` bound, v1-style — before the execution's own (empty) output:

In [ ]:
g = start_gateway()
os.environ['CLIK_DEMO'] = 'via-defaults'
gw = await Gateway(g.url).initialize(session_defaults(cfgd))
banner = await gw.text('py', code='')
kid = re.search(r'created kernel (\w+)', banner).group(1)
assert 'ready, file startup.py' in banner
banner

'created kernel aace7ffccfe443bda275ec185b8f66e3 language=python\nready, file startup.py\n'

State persists across calls, the kernel was born in the conversation's cwd, and its environment came through the session defaults: `CLIK_DEMO` was set only after the scratch gateway started, so process inheritance could not have put it there. Magics run as written, which is why `%%bash` needs no tool of its own:


In [ ]:
test_eq(await gw.text('py', code='base'), '42')
test_eq(await gw.text('py', code='import os; os.getcwd()'), repr(os.getcwd()))
test_eq(await gw.text('py', code="os.environ['CLIK_DEMO']"), "'via-defaults'")
hi = await gw.text('py', code='%%bash\necho hi')
test_eq(hi, 'hi\n')
hi


'hi\n'

The three inspector behaviors hold as installed: a note prints before the cell's output, a crashing inspector is reported and fails open, and a `RuleBlock` stops the cell before anything in it runs, leaving state intact:


In [ ]:
noted = await gw.text('py', code='import time; time.sleep(0.01); 7')
assert noted.startswith('<stdout>\nnote: sleeping') and '7' in noted
buggy = await gw.text('py', code='trigger_bug = 1; 8')
assert 'inspector error (cell runs anyway)' in buggy and '8' in buggy
blocked = await gw.text('py', code='import ctypes')
assert 'no ctypes in kernels' in blocked
test_eq(await gw.text('py', code='base'), '42')
blocked

"---------------------------------------------------------------------------\nRuleBlock                                 Traceback (most recent call last)\nCell In[1], line 24, in _ClikInspect.visit(self, tree)\n     20         for f in self.fs:\n     21             try:\n     22                 note = f(tree, _clik_src) if len(_clik_inspect.signature(f).parameters) > 1 else f(tree)\n     23                 if note: print(note, end='')\n---> 24             except InputRejected: raise\n     25             except Exception as e: print(f'inspector error (cell runs anyway): {e!r}')\n     26         return tree\n\nFile inspectors.py:7, in block_ctypes(tree)\n      5 'Could not get source, probably due dynamically evaluated source code.'\n\nRuleBlock: no ctypes in kernels"

The lifecycle rules, all visible from a second session. `create` binds a kernel to a dialog name and autocloses it with its session by default; `autoclose=false` makes it a keeper. `use_kernel` attaches to any running kernel — here the first session's auto kernel — and attaching claims nothing: when the first session ends, its DELETE stops its auto kernel and its autoclose create, out from under the attachment, while the keeper lives on.

In [ ]:
assert (await gw.text('create', dlgname='demo.ipynb')).startswith('created kernel ')
kept = re.search(r'kernel (\w+)', await gw.text('create', dlgname='keeper.ipynb', autoclose=False)).group(1)

gw2 = await Gateway(g.url).initialize(session_defaults(cfgd))
assert kid[:8] in await gw2.text('use_kernel', kernel=kid[:8])
test_eq(await gw2.text('py', code='base'), '42')

await gw.aclose()
listing = await gw2.text('list_kernels')
assert kid not in listing and 'demo.ipynb' not in listing and kept in listing
listing

'7e5c70fed1ce4f84a1458878b3054af1  alive  language=python  connections=0  dlgname=keeper.ipynb'

`restart` gives a fresh interpreter under the same id, and the gateway re-runs this session's startup in a kernel the session created — so the banner returns and `base` is back, while everything else is gone. A broken config file is the other side of the same contract: startup errors stop the fresh kernel and fail the creating call, so a kernel never runs half-inspected.

In [ ]:
assert 'created kernel' in await gw2.text('create', dlgname='r.ipynb')
await gw2.text('py', code='y = 5')
res = await gw2.text('restart')
assert res.startswith('restarted kernel ') and 'ready' in res
test_eq(await gw2.text('py', code='base'), '42')
assert 'NameError' in await gw2.text('py', code='y')
res

'restarted kernel 8f2980ce255542f08b7780c14a73a1bfready, file startup.py\n'

In [ ]:
bad = Path(tempfile.mkdtemp())
(bad/'inspectors.py').write_text('import not_a_module')
gw3 = await Gateway(g.url).initialize(session_defaults(bad))
before = await gw2.text('list_kernels')
with expect_fail(RuntimeError, contains='startup failed'): await gw3.text('py', code='1')
after = await gw2.text('list_kernels')
test_eq(after, before)
after

'7e5c70fed1ce4f84a1458878b3054af1  alive  language=python  connections=0  dlgname=keeper.ipynb\n8f2980ce255542f08b7780c14a73a1bf  alive  language=python  connections=1  dlgname=r.ipynb  <- current'

Some hosts should not see startup output at all. `quiet=True` in the defaults keeps it out of every reply — the kernel id line stays, a protocol fact — and stamps `CLIKERNEL_QUIET=1` into the kernel's environment so kernel-side tooling can hush narration of its own:

In [ ]:
gwq = await Gateway(g.url).initialize(session_defaults(cfgd, quiet=True))
qbanner = await gwq.text('py', code='')
assert 'created kernel' in qbanner and 'ready' not in qbanner
test_eq(await gwq.text('py', code='base'), '42')
test_eq(await gwq.text('py', code="import os; os.environ['CLIKERNEL_QUIET']"), "'1'")
await gwq.aclose()
qbanner

'created kernel d01289c229b64075827143dc6d1f1d87 language=python\n'

## The default gateway

A conversation needs a local gateway before its first reply to the harness, so `default_gateway` makes one exist: probe the default URL, and when nothing answers, start an owned rustygate child on a free port. The distinction is ownership, not configuration. A gateway that was already running is somebody's — kernels on it outlive the conversation, and that is the resume story. An owned child lives exactly as long as the conversation, so nothing on it survives; a conversation that wants a kernel to persist needs a gateway that persists, which is one `rustygate` service away.

In [ ]:
#| export
async def default_gateway(
    cfgdir=None,  # Config dir for `session_defaults` (the standard one if None)
    quiet=False,  # Keep startup output out of replies?
):
    "An initialized `Gateway` on the default local gateway, plus the owned child rustygate when none was running (else None)"
    url, token, verify = resolve('', cfgdir)
    d = session_defaults(cfgdir, quiet)
    try: return await Gateway(url, token, verify).initialize(d), None
    except httpx.ConnectError:
        child = start_gateway()
        return await Gateway(child.url).initialize(d), child

Both cases live. Pointed at the running scratch gateway, the probe answers and no child starts:


In [ ]:
os.environ['CLIKERNEL_HOST'] = g.url
gf, child = await default_gateway(cfgd)
assert child is None and gf.url == g.url
await gf.aclose()
gf.url


'http://127.0.0.1:58034'

Pointed at a port where nothing listens, an owned child appears, serves the same session shape, and is gone once stopped:

In [ ]:
os.environ['CLIKERNEL_HOST'] = 'http://127.0.0.1:1'
go, child = await default_gateway(cfgd)
assert child is not None and go.url == child.url
banner = await go.text('py', code='')
assert 'created kernel' in banner and 'ready' in banner
await go.aclose()
child.stop()
child.url

'http://127.0.0.1:58133'

In [ ]:
#|hide
await gw2.aclose()
g.stop()
for k in ('CLIKERNEL_HOST', 'CLIK_DEMO'): os.environ.pop(k, None)

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()